In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder.appName("Observations-Silver").getOrCreate()

In [0]:
df = spark.table("philippinebiodiveristyplatform.bronze.observations")

max_lat = 18.5052
min_lat = 5.5810
max_long = 126.5374
min_long = 117.1743

#Keep only research grade (meaning more than one validator agreed), have taxon id and in ph (double check for bronze)
df = df.filter((F.col("quality_grade") == "research") & F.col("taxon_id").isNotNull() & (F.col("latitude").between(min_lat, max_lat)
    & F.col("longitude").between(min_long, max_long)) & F.col("observation_uuid").isNotNull())

#Deduplicate, keeps the latest record
window_spec = Window.partitionBy("observation_uuid").orderBy(
    F.col("observed_on").desc_nulls_last(),
    F.col("ingestion_timestamp").desc_nulls_last(),
)
deduped_df = (
    df.withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Schema Enforcement
deduped_df = (
    deduped_df.withColumn("observation_uuid", F.col("observation_uuid").cast("string"))
    .withColumn("observer_id", F.col("observer_id").cast("integer"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("positional_accuracy", F.col("positional_accuracy").cast("integer"))
    .withColumn("taxon_id", F.col("taxon_id").cast("integer"))
    .withColumn("quality_grade", F.col("quality_grade").cast("string"))
    .withColumn("observed_on", F.col("observed_on").cast("timestamp"))
    .withColumn("anomaly_score", F.col("anomaly_score").cast("double"))
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
    .withColumn("source_file_path", F.col("source_file_path").cast("string"))
)

# Group them into ranges for later
acc = F.col("positional_accuracy")
anom = F.col("anomaly_score")

accuracy_flag = (
    F.when(acc.isNull(), "unknown")
    .when(acc < 100, "high_precision")
    .when(acc < 1000, "moderate_precision")
    .when(acc < 10000, "low_precision")
    .otherwise("very_low_or_obscured")
)

anomaly_flag = (
    F.when(anom.isNull(), "unknown")
    .when(anom < 1.0, "low")
    .when(anom < 6.0, "moderate")
    .when(anom < 10.0, "elevated")
    .otherwise("high")
)

deduped_df = (
    deduped_df.withColumn("accuracy_flag", accuracy_flag)
    .withColumn("anomaly_flag", anomaly_flag)
)

deduped_df.write.format("delta").mode("overwrite").saveAsTable("philippinebiodiveristyplatform.silver.observations")


